# Multivariate Physicochemical and Biological Time Series from Estuarine and Coastal Stations in the Basque Country (1995–2014) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.3frn-j6jz/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Extract metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display available record sets, fields, and columns by @id
record_sets = dataset.record_sets
print(f"Record sets found: {[rs['@id'] for rs in record_sets]}")

# Show fields and columns for each record set
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', '')}")
            # Check for columns
            if 'column' in field:
                columns = field['column'] if isinstance(field['column'], list) else [field['column']]
                for col in columns:
                    print(f"    Column @id: {col['@id']}, label: {col.get('label', '')}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# List columns of each DataFrame, referencing field and column @id
for rs_id, df in dataframes.items():
    print(f"Columns for record set @id {rs_id}: {df.columns.tolist()}")

# Show head of the first available record set
if dataframes:
first_rs_id = next(iter(dataframes))
print(f"Sample data from record set @id {first_rs_id}:")
dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set and numeric field for EDA
# For demonstration, use the first available record set
record_set_id = first_rs_id
df = dataframes[record_set_id]

# Identify a numeric column by @id (column name in DataFrame)
numeric_col = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_col = col
        break
if numeric_col is None:
    numeric_col = df.columns[0]  # fallback to first column if not found
numeric_field_id = numeric_col  # all columns are referenced as @id

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
norm_col_name = f"{numeric_field_id}_normalized"
filtered_df[norm_col_name] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col_name]].head())

# Group by a key field if available
group_field = None
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization for numeric field distribution
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Scatter plot if there are at least two numeric columns
numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if len(numeric_cols) >= 2:
    plt.figure(figsize=(6, 6))
    sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
    plt.title(f"Scatter plot of {numeric_cols[0]} vs {numeric_cols[1]}")
    plt.xlabel(numeric_cols[0])
    plt.ylabel(numeric_cols[1])
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading and exploring the FAIR² dataset using the `mlcroissant` library.
- All dataset entities (record sets, fields, columns) are referenced by their `@id`.
- Data extraction, filtering, normalization, and grouping by key attributes were performed for basic exploratory analysis.
- Visualizations displayed distributions and relationships between numeric fields, informing potential further analysis and research directions.

You can use this notebook template to extend analyses by referencing additional record sets and fields via their `@id`, and applying domain-specific processing relevant to ecological or environmental datasets.